In [2]:
import warnings
warnings.simplefilter('ignore')

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, confusion_matrix

In [3]:
data = pd.read_csv("Data/heart.csv")

In [4]:
data

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2,1
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2,0
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2,1


In [5]:
data.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


In [7]:
x = data.drop('target',axis=1)

In [8]:
y = data['target']

In [9]:
x

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2


In [10]:
y

0       0
1       0
2       0
3       0
4       0
       ..
1020    1
1021    0
1022    0
1023    1
1024    0
Name: target, Length: 1025, dtype: int64

#### Không cần thiết scaling bởi vì toàn bộ dữ liệu là numerical data và không có categorical data nào cả

In [11]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [12]:
print(x.shape)
print(x_train.shape)
print(x_test.shape)
print()
print(y.shape)
print(y_train.shape)
print(y_test.shape)

(1025, 13)
(820, 13)
(205, 13)

(1025,)
(820,)
(205,)


In [13]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from sklearn.pipeline import Pipeline

In [16]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000))
    ]),

    'Naive Bayes': Pipeline([
        ('scaler', StandardScaler()),
        ('model', GaussianNB())
    ]),

    'Random Forest': RandomForestClassifier(
        random_state=42
    ),

    'XGBoost': XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        n_jobs=1
    ),

    'K Nearest Neighbors': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier())
    ]),

    'Decision Tree': DecisionTreeClassifier(
        random_state=42
    ),

    'Support Vector Machine': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(
            kernel='rbf',
            probability=True,
            random_state=42
        ))
    ])
}

In [20]:
#Khai báo params
param_grids = {
    'Logistic Regression': {
        'model__C': [0.01, 0.1, 1, 10, 100],
        'model__solver': ['lbfgs', 'liblinear'],
        'model__class_weight': [None, 'balanced']
    },

    'Naive Bayes': {
        'model__var_smoothing': np.logspace(-12, -7, 6)
    },

    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 3, 5, 8, 12],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2'],
        'class_weight': [None, 'balanced']
    },

    'XGBoost': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [2, 3, 5, 7],
        'min_child_weight': [1, 3, 5],
        'subsample': [0.7, 0.9, 1.0],
        'colsample_bytree': [0.7, 0.9, 1.0],
        'reg_lambda': [1, 3, 5]
    },

    'K Nearest Neighbors': {
        'model__n_neighbors': [3, 5, 7, 9, 11, 15],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2]
    },

    'Decision Tree': {
        'criterion': ['gini', 'entropy', 'log_loss'],
        'max_depth': [None, 3, 5, 7, 10],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'class_weight': [None, 'balanced']
    },

    'Support Vector Machine': {
        'model__C': [0.1, 1, 2, 10, 100],
        'model__gamma': ['scale', 'auto', 0.01, 0.1, 1],
        'model__class_weight': [None, 'balanced']
    }
}
#Các tham số có tiền tố model__ là tham số của estimator nằm trong Pipeline ở bước tên model. Các tham số khác là tham số của estimator được khai báo trực tiếp.
#Model đặt trong Pipeline thì estimator là pipeline còn ngoài thì estimator là model. Vì vậy, khi khai báo tham số cho model trong pipeline thì phải thêm tiền tố model__ trước tên tham số của model.

In [21]:
#Thiết lập Cross-Validation và chạy GridSearchCV cho từng model
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

best_models = {}
cv_results = []

for name, model in models.items():

    print(f'\nTuning {name}...')

    grid_search = GridSearchCV( #RandomizedSearchCV cũng tương tự nhưng thay vì thử tất cả các kết hợp tham số thì nó sẽ chọn ngẫu nhiên một số kết hợp tham số để thử. Điều này giúp tiết kiệm thời gian tính toán khi không cần thiết phải thử tất cả các kết hợp tham số.
        estimator=model,
        param_grid=param_grids[name],
        scoring='roc_auc',
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=1,
        return_train_score=True
    )

    grid_search.fit(x_train, y_train)

    best_models[name] = grid_search.best_estimator_

    cv_results.append({
        'Model': name,
        'Best CV ROC-AUC': grid_search.best_score_,
        'Best Params': grid_search.best_params_
    })

    print('Best CV ROC-AUC:', grid_search.best_score_)
    print('Best Params:', grid_search.best_params_)


Tuning Logistic Regression...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best CV ROC-AUC: 0.9273711794844519
Best Params: {'model__C': 1, 'model__class_weight': None, 'model__solver': 'lbfgs'}

Tuning Naive Bayes...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best CV ROC-AUC: 0.9171016824451298
Best Params: {'model__var_smoothing': np.float64(1e-12)}

Tuning Random Forest...
Fitting 5 folds for each of 540 candidates, totalling 2700 fits
Best CV ROC-AUC: 0.9966642068219693
Best Params: {'class_weight': 'balanced', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}

Tuning XGBoost...
Fitting 5 folds for each of 2916 candidates, totalling 14580 fits
Best CV ROC-AUC: 0.9944305791937028
Best Params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'min_child_weight': 1, 'n_estimators': 100, 'reg_lambda': 1, 'subsample': 1.0}

Tuning K Nearest Neighbors...
Fitting 5 folds for each 

In [22]:
#Tổng hợp kết quả Cross-Validation và hiển thị
cv_results_df = pd.DataFrame(cv_results)
cv_results_df

,Model,Best CV ROC-AUC,Best Params
0,Logistic Regression,0.927371,"{'model__C': 1, 'model__class_weight': None, '..."
1,Naive Bayes,0.917102,{'model__var_smoothing': 1e-12}
2,Random Forest,0.996664,"{'class_weight': 'balanced', 'max_depth': None..."
3,XGBoost,0.994431,"{'colsample_bytree': 1.0, 'learning_rate': 0.1..."
4,K Nearest Neighbors,0.995295,"{'model__n_neighbors': 15, 'model__p': 2, 'mod..."
5,Decision Tree,0.982295,"{'class_weight': None, 'criterion': 'gini', 'm..."
6,Support Vector Machine,0.998779,"{'model__C': 0.1, 'model__class_weight': None,..."


In [26]:
cv_results_df = cv_results_df.sort_values(
    by='Best CV ROC-AUC',
    ascending=False
).reset_index(drop=True)

print(cv_results_df.to_string(index=False))

                 Model  Best CV ROC-AUC                                                                                                                                    Best Params
Support Vector Machine         0.998779                                                                              {'model__C': 0.1, 'model__class_weight': None, 'model__gamma': 1}
         Random Forest         0.996664    {'class_weight': 'balanced', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
   K Nearest Neighbors         0.995295                                                                        {'model__n_neighbors': 15, 'model__p': 2, 'model__weights': 'distance'}
               XGBoost         0.994431 {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'min_child_weight': 1, 'n_estimators': 100, 'reg_lambda': 1, 'subsample': 1.0}
         Decision Tree         0.982295                                  {'class_weig

In [27]:
cv_results_df.to_csv('cv_results.csv', index=False)